In [54]:
from langchain_core.tools import tool
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: float, conversion_rate: float) -> float:
    """
    Converts a currency value using a conversion rate.
    Use the rate returned by get_conversion_factor as the conversion_rate argument.
    """
    return base_currency_value * conversion_rate

convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'number'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [55]:
result =get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})
print(result)

{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1772582402, 'time_last_update_utc': 'Wed, 04 Mar 2026 00:00:02 +0000', 'time_next_update_unix': 1772668802, 'time_next_update_utc': 'Thu, 05 Mar 2026 00:00:02 +0000', 'base_code': 'USD', 'target_code': 'INR', 'conversion_rate': 92.1093}


In [56]:
convert.invoke({'base_currency_value':5.99, 'conversion_rate':result['conversion_rate']})

551.7347070000001

In [57]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

from langchain_core.messages import HumanMessage


In [58]:
# tool binding
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]
messages


[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [59]:
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

In [60]:
ai_message.tool_calls
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

messages


[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 128, 'total_tokens': 150, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DFVa13Lpz7FFHjtFcrRS3rnQ8QYxH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cb684-d6a0-7c22-8b59-d18716d458fe-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': 'call_lzxgAYxQjLKxmPegFoCvOzJd', 'type': 'tool_call'}], invalid_t

In [61]:
llm_with_tools.invoke(messages).content

''

In [62]:
from langchain.agents import create_agent  # new import

agent_executor = create_agent(
    model=llm,
    tools=[get_conversion_factor, convert]
)

In [63]:
from langchain.agents import create_agent

# Create agent
agent_executor = create_agent(
    model=llm,
    tools=[get_conversion_factor, convert]
)

# Run agent
response = agent_executor.invoke({
    "messages": [("human", "Convert 5.99 USD to INR")]
})

print(response["messages"][-1].content)

5.99 USD is approximately 551.73 INR.


In [64]:
response = agent_executor.invoke({
    "messages": [("human", "Convert 5.99 USD to INR")]
}, debug=True)  # add debug=True to see tool calls

[values] {'messages': [HumanMessage(content='Convert 5.99 USD to INR', additional_kwargs={}, response_metadata={}, id='b4879908-3af0-4ef5-bf41-1f0fa6334be3')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 113, 'total_tokens': 135, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DFVa7r65RmFzQi6TtdItzC7mWD5AE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cb684-edeb-7901-9a37-9f4d637e0093-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_7deDPHs53QPhmTzq3nb1rg9M', 'type': 